In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import (
    make_column_transformer,
    make_column_selector,
    TransformedTargetRegressor,
)
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

import optuna

import xgboost as xgb
import lightgbm as lgb

from model import Regressor
from utils import xgb_objective, lgb_objective
import torch
import torch.nn as nn
import torch.optim as optim

from tqdm import tqdm

from joblib import dump

c:\Users\bcong\Dropbox\github\cta-ridership-prediction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
cta_df = pd.read_parquet(
    "../feature_engineer/output/cta_ridership_with_features.parquet"
)
cta_df = cta_df.reset_index(drop=True)

In [3]:
cta_df.head(10)

,station_id,stationname,date,daytype,rides,map_id,red,blue,g,brn,...,o,location,lat,lon,line,year,month,day,day_of_week_num,day_of_week_name
0,40350,UIC-Halsted,2001-01-01,U,273,40350,False,True,False,False,...,False,"{""latitude"":""41.875474"",""longitude"":""-87.64970...",41.875474,-87.649707,blue,2001,1,1,0,Monday
1,41130,Halsted-Orange,2001-01-01,U,306,41130,False,False,False,False,...,True,"{""latitude"":""41.84678"",""longitude"":""-87.648088...",41.846780,-87.648088,orange,2001,1,1,0,Monday
2,40760,Granville,2001-01-01,U,1059,40760,True,False,False,False,...,False,"{""latitude"":""41.993664"",""longitude"":""-87.65920...",41.993664,-87.659202,red,2001,1,1,0,Monday
3,40070,Jackson/Dearborn,2001-01-01,U,649,40070,False,True,False,False,...,False,"{""latitude"":""41.878183"",""longitude"":""-87.62929...",41.878183,-87.629296,blue,2001,1,1,0,Monday
4,40090,Damen-Brown,2001-01-01,U,411,40090,False,False,False,True,...,False,"{""latitude"":""41.966286"",""longitude"":""-87.67863...",41.966286,-87.678639,brown,2001,1,1,0,Monday
5,40590,Damen/Milwaukee,2001-01-01,U,870,40590,False,True,False,False,...,False,"{""latitude"":""41.909744"",""longitude"":""-87.67743...",41.909744,-87.677437,blue,2001,1,1,0,Monday
6,40720,East 63rd-Cottage Grove,2001-01-01,U,391,40720,False,False,True,False,...,False,"{""latitude"":""41.780309"",""longitude"":""-87.60585...",41.780309,-87.605857,green,2001,1,1,0,Monday
7,41260,Austin-Lake,2001-01-01,U,399,41260,False,False,True,False,...,False,"{""latitude"":""41.887293"",""longitude"":""-87.77413...",41.887293,-87.774135,green,2001,1,1,0,Monday
8,40230,Cumberland,2001-01-01,U,788,40230,False,True,False,False,...,False,"{""latitude"":""41.984246"",""longitude"":""-87.83802...",41.984246,-87.838028,blue,2001,1,1,0,Monday
9,41120,35-Bronzeville-IIT,2001-01-01,U,448,41120,False,False,True,False,...,False,"{""latitude"":""41.831677"",""longitude"":""-87.62582...",41.831677,-87.625826,green,2001,1,1,0,Monday


# Pre-processing

## Encode categorical features

In [4]:
ordinal_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
onehot_encoder = OneHotEncoder()
scaler = StandardScaler()

preprocessor = make_column_transformer(
    # (ordinal_encoder, make_column_selector(dtype_include=object)),
    (onehot_encoder, make_column_selector(dtype_include=object)),
    (scaler, make_column_selector(dtype_include="number")),
    remainder="passthrough",
)

In [5]:
X = preprocessor.fit_transform(
    cta_df[
        [
            "line",
            "year",
            "month",
            "day",
            "day_of_week_num",
            "day_of_week_name",
            "lat",
            "lon",
        ]
    ]
)
y = cta_df["rides"]

# Train-test split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42
)
y_train = y_train.astype(float)
y_test = y_test.astype(float)

## Save test data

In [7]:
import scipy.sparse as sp

feature_names = preprocessor.get_feature_names_out()
X_test_dense = X_test.toarray() if sp.issparse(X_test) else X_test

test_df = pd.DataFrame(X_test_dense, columns=feature_names)
test_df["rides"] = y_test.values

test_df.to_parquet("output/cta_ridership_test.parquet", index=False)

In [8]:
test_df.size

9223764

# Models

Let's do a first pass without any hyperparameter tuning to get a sense of which models have the most predictive power.

## OLS

In [8]:
mod_ols = TransformedTargetRegressor(
    regressor=LinearRegression(), func=np.log1p, inverse_func=np.expm1
)

mod_ols.fit(X_train, y_train)
y_pred_ols = mod_ols.predict(X_test)
print("R squared: ", r2_score(y_test, y_pred_ols))

R squared:  0.13097915308938846


## RF

In [9]:
mod_rf = TransformedTargetRegressor(
    regressor=RandomForestRegressor(n_estimators=10, random_state=42),
    func=np.log1p,
    inverse_func=np.expm1,
)

mod_rf.fit(X_train, y_train)
y_pred_rf = mod_rf.predict(X_test)
print("R squared: ", r2_score(y_test, y_pred_rf))

R squared:  0.9678180953995276


## XGBoost

In [10]:
mod_xgb = TransformedTargetRegressor(
    regressor=xgb.XGBRegressor(random_state=42, tree_method="hist"),
    func=np.log1p,
    inverse_func=np.expm1,
)

mod_xgb.fit(X_train, y_train)
y_pred_xgb = mod_xgb.predict(X_test)
print("R squared: ", r2_score(y_test, y_pred_xgb))

R squared:  0.9379040151884488


## Neural network using PyTorch

In [11]:
torch.manual_seed(42)

# Preprocess ----
# Log transform
y_train_log = np.log1p(y_train).values.reshape(-1, 1)
y_test_log = np.log1p(y_test).values.reshape(-1, 1)

y_train_scaled = scaler.fit_transform(y_train_log)
y_test_scaled = scaler.transform(y_test_log)

# Initialize
model = Regressor(n_in=X_train.shape[1])
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Train
X_train_tensor = torch.from_numpy(X_train).float()
y_train_tensor = torch.from_numpy(y_train_scaled).float()

for epoch in tqdm(range(500)):
    optimizer.zero_grad()
    preds = model(X_train_tensor)
    loss = criterion(preds, y_train_tensor)
    loss.backward()
    optimizer.step()

# Evaluate
model.eval()

with torch.no_grad():
    y_pred_scaled = model(torch.from_numpy(X_test).float()).numpy()
    y_pred = np.expm1(scaler.inverse_transform(y_pred_scaled))
    r2_nn = r2_score(y_test, y_pred)
    print("R squared: ", r2_nn)

100%|██████████| 500/500 [05:00<00:00,  1.67it/s]

R squared:  0.37820722970516496


## LightGBM

In [12]:
mod_lgb = TransformedTargetRegressor(
    regressor=lgb.LGBMRegressor(random_state=42), func=np.log1p, inverse_func=np.expm1
)

mod_lgb.fit(X_train, y_train)
y_pred_lgb = mod_lgb.predict(X_test)
print("R squared: ", r2_score(y_test, y_pred_lgb))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007948 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 399
[LightGBM] [Info] Number of data points in the train set: 851226, number of used features: 21
[LightGBM] [Info] Start training from score 7.452894


c:\Users\bcong\Dropbox\github\cta-ridership-prediction\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


R squared:  0.8845346834776889


## Tuned XGB

In [20]:
y_log = np.log1p(y.astype(float))

study = optuna.create_study(
    direction="maximize", sampler=optuna.samplers.TPESampler(seed=42)
)
study.optimize(lambda trial: xgb_objective(trial, X, y_log), n_trials=20)

[I 2026-03-10 15:23:39,861] A new study created in memory with name: no-name-ef915283-00c7-4236-af4d-ac7a25d8c380
[I 2026-03-10 15:24:06,629] Trial 0 finished with value: 0.5923483802379389 and parameters: {'n_estimators': 137, 'max_depth': 10}. Best is trial 0 with value: 0.5923483802379389.
[I 2026-03-10 15:24:33,010] Trial 1 finished with value: 0.585636234416549 and parameters: {'n_estimators': 173, 'max_depth': 7}. Best is trial 0 with value: 0.5923483802379389.
[I 2026-03-10 15:24:44,561] Trial 2 finished with value: 0.5515838888040829 and parameters: {'n_estimators': 115, 'max_depth': 4}. Best is trial 0 with value: 0.5923483802379389.
[I 2026-03-10 15:25:01,531] Trial 3 finished with value: 0.5817286221763507 and parameters: {'n_estimators': 105, 'max_depth': 9}. Best is trial 0 with value: 0.5923483802379389.
[I 2026-03-10 15:25:22,630] Trial 4 finished with value: 0.5702627965809887 and parameters: {'n_estimators': 160, 'max_depth': 8}. Best is trial 0 with value: 0.592348380

In [21]:
print(f"Best params is {study.best_params} with value {study.best_value}")

Best params is {'n_estimators': 169, 'max_depth': 6} with value 0.6022204297957902


In [22]:
# Predict using the best set of hyperparameters
mod_xgb_tuned = TransformedTargetRegressor(
    regressor=xgb.XGBRegressor(**study.best_params),
    func=np.log1p,
    inverse_func=np.expm1,
)

mod_xgb_tuned.fit(X_train, y_train)
y_pred_xgb_tuned = mod_xgb_tuned.predict(X_test)
print("R squared: ", r2_score(y_test, y_pred_xgb_tuned))

R squared:  0.9465992471691096


## Tuned LGB

In [23]:
study_lgb = optuna.create_study(
    direction="maximize", sampler=optuna.samplers.TPESampler(seed=42)
)
study_lgb.optimize(lambda trial: lgb_objective(trial, X, y_log), n_trials=10)

[I 2026-03-10 15:30:39,449] A new study created in memory with name: no-name-e6ef50f8-8fbb-4273-839e-f2627e092103


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006473 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 391
[LightGBM] [Info] Number of data points in the train set: 846992, number of used features: 21
[LightGBM] [Info] Start training from score 7.457564


c:\Users\bcong\Dropbox\github\cta-ridership-prediction\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008644 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 392
[LightGBM] [Info] Number of data points in the train set: 846992, number of used features: 21
[LightGBM] [Info] Start training from score 7.296194


c:\Users\bcong\Dropbox\github\cta-ridership-prediction\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


...

c:\Users\bcong\Dropbox\github\cta-ridership-prediction\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009352 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 392
[LightGBM] [Info] Number of data points in the train set: 846992, number of used features: 21
[LightGBM] [Info] Start training from score 7.296194
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


c:\Users\bcong\Dropbox\github\cta-ridership-prediction\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007834 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 387
[LightGBM] [Info] Number of data points in the train set: 846992, number of used features: 21
[LightGBM] [Info] Start training from score 7.603210
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


c:\Users\bcong\Dropbox\github\cta-ridership-prediction\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[I 2026-03-10 15:32:27,278] Trial 9 finished with value: 0.6140091771083629 and parameters: {'n_estimators': 143, 'max_depth': 5}. Best is trial 1 with value: 0.6269965116753835.


In [26]:
print(
    f"Best params for LGB is {study_lgb.best_params} with value {study_lgb.best_value}"
)

Best params for LGB is {'n_estimators': 173, 'max_depth': 7} with value 0.6269965116753835


In [27]:
# Predict using the best set of hyperparameters
mod_lgb_tuned = TransformedTargetRegressor(
    regressor=lgb.LGBMRegressor(**study_lgb.best_params),
    func=np.log1p,
    inverse_func=np.expm1,
)

mod_lgb_tuned.fit(X_train, y_train)
y_pred_lgb_tuned = mod_lgb_tuned.predict(X_test)
print("R squared: ", r2_score(y_test, y_pred_lgb_tuned))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.065554 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 399
[LightGBM] [Info] Number of data points in the train set: 851226, number of used features: 21
[LightGBM] [Info] Start training from score 7.452894


c:\Users\bcong\Dropbox\github\cta-ridership-prediction\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


R squared:  0.9112887840234672


## Dump joblib model

In [ ]:
dump(mod_rf, "output/mod_rf.joblib")